# Preprocessing — UAV VoiceControl Benchmark

Dieses Notebook führt das **Upsampling** der unterrepräsentierten Klassen `forward` und `backward` durch — mittels **Data Augmentation** — sowie das **Downsampling** der überrepräsentierten Klasse `Unknown Words`.

**Warum Augmentation statt einfachem Kopieren?**  
Würden wir dieselbe Datei einfach mehrfach verwenden, würde das Modell exakt identische Samples mehrfach sehen → **Overfitting** auf diese Samples. Augmentierung erstellt stattdessen neue, leicht veränderte Varianten, die reale Bedingungen simulieren.

**Angewendete Techniken:**
- **Pitch Shift** — Tonhöhe leicht verschieben (simuliert unterschiedliche Stimmlagen)
- **Time Shift** — Audio zeitlich verschieben (simuliert unterschiedliches Timing im Clip)
- **Random Downsampling** — zufällige Auswahl einer Teilmenge der Unknown Words

**Inhalt:**
1. Imports
2. Pfade & Konfiguration
3. Zielanzahl berechnen (Durchschnitt aller Keywords)
4. Augmentierungs-Funktionen
5. Upsampling Loop (forward & backward)
6. Verifikation Upsampling
7. Downsampling Unknown Words
8. Verifikation Downsampling

## 1. Imports

In [1]:
import os
import random
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path

random.seed(42)
np.random.seed(42)

## 2. Pfade & Konfiguration

In [2]:
base_path      = Path.cwd().parent
raw_data_path  = base_path / 'data' / 'raw' / 'speech_command_data'
aug_data_path  = base_path / 'data' / 'augmented'

KEY_WORDS      = ['forward', 'backward', 'up', 'down', 'stop', 'right', 'left', 'go', 'one', 'two']
CLASSES_TO_AUG = ['forward', 'backward']   # nur diese werden upgesampelt
SAMPLE_RATE    = 16000

# Ausgabe-Ordner anlegen
for cls in CLASSES_TO_AUG:
    (aug_data_path / cls).mkdir(parents=True, exist_ok=True)

print(f'Raw-Data-Pfad  : {raw_data_path}')
print(f'Augmented-Pfad : {aug_data_path}')

Raw-Data-Pfad  : /Users/kayraozturk/Desktop/HRW/eingebettete_systeme_2/UAV_VoiceControl_Benchmark/data/raw/speech_command_data
Augmented-Pfad : /Users/kayraozturk/Desktop/HRW/eingebettete_systeme_2/UAV_VoiceControl_Benchmark/data/augmented


## 3. Zielanzahl berechnen

Als Ziel nehmen wir den **Durchschnitt aller 10 Keyword-Klassen** aus den Raw-Daten.  
So heben wir `forward` und `backward` auf ein repräsentatives Niveau — ohne die anderen Klassen zu benachteiligen.

In [3]:
file_counts = {}
for cls in KEY_WORDS:
    cls_path = raw_data_path / cls
    file_counts[cls] = len(list(cls_path.glob('*.wav')))

target_count = int(np.mean(list(file_counts.values())))

print('Dateianzahl pro Keyword-Klasse (Raw):')
for cls, count in sorted(file_counts.items(), key=lambda x: x[1]):
    marker = ' ← wird upgesampelt' if cls in CLASSES_TO_AUG else ''
    print(f'  {cls:<12}: {count:>5}{marker}')

print(f'\nZielanzahl (Durchschnitt): {target_count}')
print()
for cls in CLASSES_TO_AUG:
    needed = target_count - file_counts[cls]
    print(f'  {cls}: benötigt {needed} augmentierte Samples')

Dateianzahl pro Keyword-Klasse (Raw):
  forward     :  1557 ← wird upgesampelt
  backward    :  1664 ← wird upgesampelt
  up          :  3723
  right       :  3778
  left        :  3801
  stop        :  3872
  go          :  3880
  two         :  3880
  one         :  3890
  down        :  3917

Zielanzahl (Durchschnitt): 3396

  forward: benötigt 1839 augmentierte Samples
  backward: benötigt 1732 augmentierte Samples


## 4. Augmentierungs-Funktionen

### Pitch Shift
Verschiebt die Tonhöhe um einen zufälligen Wert zwischen **-2 und +2 Halbtönen**.  
Warum ±2? Größere Verschiebungen klingen unnatürlich und könnten das Wort akustisch verfremden.

### Time Shift
Rollt das Audio-Signal um eine zufällige Anzahl Samples — der Teil der "herausfällt" wird am anderen Ende wieder angehängt.  
So bleibt die Länge konstant (1 Sekunde / 16.000 Samples), aber das Wort sitzt an einer leicht anderen Position im Clip.  
Maximale Verschiebung: **±10 %** der Gesamtlänge (±1.600 Samples).

In [4]:
def pitch_shift(audio: np.ndarray, sr: int) -> np.ndarray:
    """Tonhöhe um zufälligen Wert zwischen -2 und +2 Halbtönen verschieben."""
    n_steps = random.uniform(-2.0, 2.0)
    return librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps)


def time_shift(audio: np.ndarray, sr: int, max_shift_ratio: float = 0.1) -> np.ndarray:
    """Audio zeitlich um max ±10 % der Clip-Länge verschieben (zirkulär)."""
    max_shift = int(len(audio) * max_shift_ratio)
    shift = random.randint(-max_shift, max_shift)
    return np.roll(audio, shift)


AUGMENTATIONS = [
    ('pitch',     pitch_shift),
    ('timeshift', time_shift),
]

print('Augmentierungs-Funktionen registriert:')
for name, fn in AUGMENTATIONS:
    print(f'  - {name}: {fn.__doc__.strip()}')

Augmentierungs-Funktionen registriert:
  - pitch: Tonhöhe um zufälligen Wert zwischen -2 und +2 Halbtönen verschieben.
  - timeshift: Audio zeitlich um max ±10 % der Clip-Länge verschieben (zirkulär).


## 5. Upsampling Loop

**Ablauf pro Klasse:**
1. Alle Original-Dateien laden
2. Zufällig eine Quelldatei auswählen
3. Zufällig eine Augmentierungsmethode anwenden
4. Augmentiertes Audio auf exakt 16.000 Samples padden/trimmen
5. Als neue `.wav`-Datei in `data/augmented/<klasse>/` speichern

> **Hinweis zum Padding:** Manche Originaldateien sind kürzer als 1 Sekunde (Minimum: 0.21 s laut Exploration).  
> Nach der Augmentierung stellen wir sicher, dass jeder Output-Clip **exakt 16.000 Samples** lang ist — konsistent mit dem Rest des Datensatzes.

In [5]:
TARGET_LENGTH = SAMPLE_RATE  # 16.000 Samples = 1 Sekunde

def pad_or_trim(audio: np.ndarray, target_length: int) -> np.ndarray:
    """Audio auf exakt target_length Samples bringen: trimmen oder zero-padden."""
    if len(audio) > target_length:
        return audio[:target_length]
    elif len(audio) < target_length:
        return np.pad(audio, (0, target_length - len(audio)))
    return audio


for cls in CLASSES_TO_AUG:
    cls_raw_path = raw_data_path / cls
    cls_aug_path = aug_data_path / cls

    source_files = list(cls_raw_path.glob('*.wav'))
    needed       = target_count - len(source_files)

    print(f'\n[{cls}] Erzeuge {needed} augmentierte Samples...')

    for i in range(needed):
        # 1. Zufällige Quelldatei
        src_file = random.choice(source_files)
        audio, sr = librosa.load(src_file, sr=SAMPLE_RATE)

        # 2. Zufällige Augmentierung
        aug_name, aug_fn = random.choice(AUGMENTATIONS)
        audio_aug = aug_fn(audio, sr)

        # 3. Auf exakt 1 Sekunde normalisieren
        audio_aug = pad_or_trim(audio_aug, TARGET_LENGTH)

        # 4. Speichern
        stem     = src_file.stem
        out_name = f'{stem}_aug_{aug_name}_{i:04d}.wav'
        out_path = cls_aug_path / out_name
        sf.write(out_path, audio_aug, SAMPLE_RATE)

        if (i + 1) % 200 == 0 or (i + 1) == needed:
            print(f'  {i + 1}/{needed} gespeichert...')

    print(f'  Fertig! Augmentierte Dateien in: {cls_aug_path}')


[forward] Erzeuge 1839 augmentierte Samples...


/Users/kayraozturk/Desktop/HRW/eingebettete_systeme_2/UAV_VoiceControl_Benchmark/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  200/1839 gespeichert...
  400/1839 gespeichert...
  600/1839 gespeichert...
  800/1839 gespeichert...
  1000/1839 gespeichert...
  1200/1839 gespeichert...
  1400/1839 gespeichert...
  1600/1839 gespeichert...
  1800/1839 gespeichert...
  1839/1839 gespeichert...
  Fertig! Augmentierte Dateien in: /Users/kayraozturk/Desktop/HRW/eingebettete_systeme_2/UAV_VoiceControl_Benchmark/data/augmented/forward

[backward] Erzeuge 1732 augmentierte Samples...
  200/1732 gespeichert...
  400/1732 gespeichert...
  600/1732 gespeichert...
  800/1732 gespeichert...
  1000/1732 gespeichert...
  1200/1732 gespeichert...
  1400/1732 gespeichert...
  1600/1732 gespeichert...
  1732/1732 gespeichert...
  Fertig! Augmentierte Dateien in: /Users/kayraozturk/Desktop/HRW/eingebettete_systeme_2/UAV_VoiceControl_Benchmark/data/augmented/backward


## 6. Verifikation Upsampling

Abschließend prüfen wir:
- Wie viele Dateien sind jetzt in `data/augmented/`?
- Raw + Augmented zusammen: erreichen wir die Zielanzahl?
- Sind die augmentierten Dateien tatsächlich 1 Sekunde lang?

In [6]:
import wave

print(f'{'Klasse':<12} | {'Raw':>6} | {'Augmented':>10} | {'Gesamt':>8} | {'Ziel':>6} | {'OK?':>4}')
print('-' * 60)

for cls in CLASSES_TO_AUG:
    raw_count = len(list((raw_data_path / cls).glob('*.wav')))
    aug_count = len(list((aug_data_path / cls).glob('*.wav')))
    total     = raw_count + aug_count
    ok        = '✓' if total >= target_count else '✗'
    print(f'{cls:<12} | {raw_count:>6} | {aug_count:>10} | {total:>8} | {target_count:>6} | {ok:>4}')

print()

# Stichprobe: Sample Rate & Länge der augmentierten Dateien prüfen
print('Stichprobe — Längen & Sample Rate der augmentierten Dateien:')
for cls in CLASSES_TO_AUG:
    aug_files = list((aug_data_path / cls).glob('*.wav'))[:5]
    print(f'  [{cls}]')
    for f in aug_files:
        with wave.open(str(f), 'r') as wf:
            frames = wf.getnframes()
            sr     = wf.getframerate()
            dur    = frames / sr
        print(f'    {f.name:<50} | {sr} Hz | {dur:.3f}s')

Klasse       |    Raw |  Augmented |   Gesamt |   Ziel |  OK?
------------------------------------------------------------
forward      |   1557 |       1839 |     3396 |   3396 |    ✓
backward     |   1664 |       1732 |     3396 |   3396 |    ✓

Stichprobe — Längen & Sample Rate der augmentierten Dateien:
  [forward]
    cc4f9250_nohash_0_aug_timeshift_0292.wav           | 16000 Hz | 1.000s
    f2e59fea_nohash_0_aug_pitch_0164.wav               | 16000 Hz | 1.000s
    57152045_nohash_4_aug_timeshift_1571.wav           | 16000 Hz | 1.000s
    525eaa62_nohash_1_aug_timeshift_0104.wav           | 16000 Hz | 1.000s
    f035e2ea_nohash_0_aug_timeshift_1403.wav           | 16000 Hz | 1.000s
  [backward]
    171edea9_nohash_3_aug_pitch_0801.wav               | 16000 Hz | 1.000s
    d070ea86_nohash_1_aug_pitch_0762.wav               | 16000 Hz | 1.000s
    a55105d0_nohash_0_aug_timeshift_0460.wav           | 16000 Hz | 1.000s
    067f61e2_nohash_0_aug_pitch_1048.wav               | 16000 Hz 

## 7. Downsampling Unknown Words

Die Unknown-Words-Klasse hat **71.867 Dateien** — mehr als doppelt so viele wie die Keywords nach dem Upsampling (~34.000).  
Wir reduzieren sie per **Random Downsampling** auf die Zielanzahl (Durchschnitt der Keywords).

**Wichtig:** Wir kopieren keine Dateien — wir speichern stattdessen eine **Manifest-Datei** (`unknown_selected.txt`).  
Diese Liste enthält die Pfade der zufällig ausgewählten Dateien.  
Das Training-Skript liest später aus dieser Liste, welche Unknown-Dateien tatsächlich genutzt werden.  

**Vorteile des Manifest-Ansatzes:**
- Kein unnötiges Kopieren von 34.000 Dateien → spart Speicherplatz
- Raw-Daten bleiben vollständig unangetastet
- Die Auswahl ist durch `random.seed(42)` **reproduzierbar**

In [7]:
UNKNOWN_WORDS = [
    'learn', 'follow', 'visual', 'tree', 'bed', 'sheila', 'cat', 'happy',
    'bird', 'marvin', 'house', 'wow', 'dog', 'three', 'four', 'off',
    'eight', 'on', 'six', 'nine', 'no', 'seven', 'yes', 'zero', 'five',
]

# Zielanzahl = SUMME aller Keyword-Dateien (raw + augmented)
# Nicht der Durchschnitt pro Klasse — wir wollen 1:1 Unknown zu Keywords gesamt
total_keyword_count = sum(
    len(list((raw_data_path / cls).glob('*.wav'))) +
    len(list((aug_data_path / cls).glob('*.wav'))) if (aug_data_path / cls).exists() else
    len(list((raw_data_path / cls).glob('*.wav')))
    for cls in KEY_WORDS
)
unknown_target = total_keyword_count

# Alle Unknown-Dateien sammeln
all_unknown_files = []
for cls in UNKNOWN_WORDS:
    cls_path = raw_data_path / cls
    if cls_path.exists():
        all_unknown_files.extend(cls_path.glob('*.wav'))

all_unknown_files = [str(f) for f in all_unknown_files]
print(f'Gesamt Unknown-Dateien gefunden  : {len(all_unknown_files)}')
print(f'Gesamt Keywords (raw + aug)       : {total_keyword_count}')
print(f'Zielanzahl Unknown (1:1 zu KW)    : {unknown_target}')
print(f'Werden entfernt (nicht genutzt)   : {len(all_unknown_files) - unknown_target}')

# Zufällige Auswahl (reproduzierbar durch seed)
random.seed(42)
selected_unknown = random.sample(all_unknown_files, unknown_target)

# Manifest speichern
manifest_path = aug_data_path / 'unknown_selected.txt'
with open(manifest_path, 'w') as f:
    for path in sorted(selected_unknown):
        f.write(path + '\n')

print(f'\nManifest gespeichert : {manifest_path}')
print(f'Einträge im Manifest : {len(selected_unknown)}')

Gesamt Unknown-Dateien gefunden  : 71867
Gesamt Keywords (raw + aug)       : 37533
Zielanzahl Unknown (1:1 zu KW)    : 37533
Werden entfernt (nicht genutzt)   : 34334

Manifest gespeichert : /Users/kayraozturk/Desktop/HRW/eingebettete_systeme_2/UAV_VoiceControl_Benchmark/data/augmented/unknown_selected.txt
Einträge im Manifest : 37533


## 8. Verifikation Downsampling

Prüfen ob das Manifest korrekt ist:
- Stimmt die Anzahl der Einträge?
- Sind alle Pfade gültig (Dateien existieren tatsächlich)?
- Wie viele Klassen sind im Manifest vertreten (faire Verteilung der Unknown-Klassen)?

In [8]:
from collections import Counter

# Manifest einlesen
with open(manifest_path, 'r') as f:
    lines = [l.strip() for l in f.readlines()]

# Anzahl prüfen
print(f'Einträge im Manifest : {len(lines)}')
print(f'Zielanzahl (Keywords): {unknown_target}')
print(f'Korrekt?             : {"✓" if len(lines) == unknown_target else "✗"}')
print()

# Existenz prüfen
missing = [p for p in lines if not Path(p).exists()]
print(f'Fehlende Dateien     : {len(missing)} {"✓" if len(missing) == 0 else "← PROBLEM!"}')
print()

# Verteilung der Unknown-Klassen im Manifest
class_counts = Counter(Path(p).parent.name for p in lines)
print(f'Verteilung der Unknown-Klassen im Manifest ({len(class_counts)} Klassen):')
for cls, cnt in sorted(class_counts.items(), key=lambda x: -x[1]):
    bar = '█' * (cnt // 100)
    print(f'  {cls:<12}: {cnt:>5}  {bar}')

print()
print('=== Gesamtübersicht nach Preprocessing ===')
print(f'{"Gruppe":<22} | {"Dateien":>8}')
print('-' * 35)

total_keywords = 0
for cls in KEY_WORDS:
    raw = len(list((raw_data_path / cls).glob('*.wav')))
    aug = len(list((aug_data_path / cls).glob('*.wav'))) if (aug_data_path / cls).exists() else 0
    total_keywords += raw + aug

ratio = len(lines) / total_keywords
print(f'{"Keywords (raw+aug)":<22} | {total_keywords:>8}')
print(f'{"Unknown (Manifest)":<22} | {len(lines):>8}')
print('-' * 35)
print(f'{"Gesamt":<22} | {total_keywords + len(lines):>8}')
print(f'\nVerhältnis Unknown:Keywords = {ratio:.2f} (Ziel: ~1.00)')

Einträge im Manifest : 37533
Zielanzahl (Keywords): 37533
Korrekt?             : ✓

Fehlende Dateien     : 0 ✓

Verteilung der Unknown-Klassen im Manifest (25 Klassen):
  seven       :  2132  █████████████████████
  yes         :  2122  █████████████████████
  five        :  2117  █████████████████████
  zero        :  2052  ████████████████████
  six         :  2032  ████████████████████
  nine        :  2023  ████████████████████
  no          :  2020  ████████████████████
  on          :  2002  ████████████████████
  eight       :  1991  ███████████████████
  three       :  1987  ███████████████████
  four        :  1938  ███████████████████
  off         :  1909  ███████████████████
  wow         :  1110  ███████████
  sheila      :  1099  ██████████
  marvin      :  1097  ██████████
  house       :  1093  ██████████
  dog         :  1088  ██████████
  bed         :  1085  ██████████
  cat         :  1080  ██████████
  happy       :  1069  ██████████
  bird        :  1053  ████████

## 9. Train / Validation / Test Split

Der Google Speech Commands Datensatz liefert zwei vordefinierte Listen:
- `validation_list.txt` — 9.981 Dateien
- `testing_list.txt` — 11.005 Dateien
- Alles übrige → Training

**Warum diese Listen verwenden statt selbst zu splitten?**  
Die Listen wurden so erstellt dass kein Sprecher in mehreren Sets vorkommt (**Speaker-Independent Split**).  
Würden wir zufällig splitten, könnte dieselbe Stimme im Training und im Test auftauchen → das Modell würde  
die Stimme wiedererkennen statt das Wort — und würde im echten Einsatz schlechter abschneiden als gemessen.

**Regeln für augmentierte und synthetische Daten:**
- Augmentierte Dateien (`forward`, `backward` aus `data/augmented/`) → **nur Training**
- Unknown-Manifest-Dateien → Split nach den vordefinierten Listen
- Silence-Segmente → proportional aufgeteilt (80 / 10 / 10)

**Output:** drei CSV-Dateien in `data/splits/`
```
path, label, split
data/raw/.../forward/abc.wav, forward, train
data/augmented/forward/abc_aug_pitch_0001.wav, forward, train
...
```

In [9]:
import csv

splits_path = base_path / 'data' / 'splits'
splits_path.mkdir(parents=True, exist_ok=True)

# Vordefinierte Listen einlesen (Format: 'klasse/dateiname.wav')
def load_list(filepath):
    with open(filepath, 'r') as f:
        return set(line.strip() for line in f if line.strip())

val_set  = load_list(raw_data_path / 'validation_list.txt')
test_set = load_list(raw_data_path / 'testing_list.txt')

print(f'Validation-Liste : {len(val_set):>6} Einträge')
print(f'Test-Liste       : {len(test_set):>6} Einträge')

def get_split(cls, filename):
    """Gibt 'val', 'test' oder 'train' zurück anhand der vordefinierten Listen."""
    key = f'{cls}/{filename}'
    if key in val_set:
        return 'val'
    elif key in test_set:
        return 'test'
    return 'train'

Validation-Liste :   9981 Einträge
Test-Liste       :  11005 Einträge


In [10]:
UNKNOWN_WORDS = [
    'learn', 'follow', 'visual', 'tree', 'bed', 'sheila', 'cat', 'happy',
    'bird', 'marvin', 'house', 'wow', 'dog', 'three', 'four', 'off',
    'eight', 'on', 'six', 'nine', 'no', 'seven', 'yes', 'zero', 'five',
]

rows = []  # (path, label, split)

# ── 1. Keywords (Raw) ─────────────────────────────────────────────────────
for cls in KEY_WORDS:
    for f in (raw_data_path / cls).glob('*.wav'):
        rows.append((str(f), cls, get_split(cls, f.name)))

# ── 2. Keywords (Augmentiert) → nur Training ──────────────────────────────
for cls in CLASSES_TO_AUG:
    aug_cls_path = aug_data_path / cls
    if aug_cls_path.exists():
        for f in aug_cls_path.glob('*.wav'):
            rows.append((str(f), cls, 'train'))

# ── 3. Unknown Words (aus Manifest) ───────────────────────────────────────
with open(aug_data_path / 'unknown_selected.txt', 'r') as mf:
    unknown_paths = [l.strip() for l in mf if l.strip()]

for p in unknown_paths:
    f    = Path(p)
    cls  = f.parent.name
    rows.append((p, 'unknown', get_split(cls, f.name)))

# ── 4. Silence (aus Background Noise) → 80/10/10 ──────────────────────────
SILENCE_RMS_THRESHOLD = 0.01
SEGMENT_LENGTH        = 16000
silence_files = []

bg_noise_path = raw_data_path / '_background_noise_'
for bg_file in sorted(bg_noise_path.glob('*.wav')):
    audio, sr = librosa.load(bg_file, sr=16000)
    n_segments = len(audio) // SEGMENT_LENGTH
    for i in range(n_segments):
        seg = audio[i * SEGMENT_LENGTH:(i + 1) * SEGMENT_LENGTH]
        rms = np.sqrt(np.mean(seg ** 2))
        if rms < SILENCE_RMS_THRESHOLD:
            # Identifier: dateiname + segment-index
            silence_files.append(f'{str(bg_file)}::seg{i:04d}')

random.shuffle(silence_files)
n      = len(silence_files)
n_val  = max(1, int(n * 0.10))
n_test = max(1, int(n * 0.10))

for i, sid in enumerate(silence_files):
    if i < n_val:
        split = 'val'
    elif i < n_val + n_test:
        split = 'test'
    else:
        split = 'train'
    rows.append((sid, 'silence', split))

# ── CSV speichern ──────────────────────────────────────────────────────────
csv_path = splits_path / 'dataset.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['path', 'label', 'split'])
    writer.writerows(rows)

print(f'CSV gespeichert: {csv_path}')
print(f'Gesamt Einträge: {len(rows)}')

CSV gespeichert: /Users/kayraozturk/Desktop/HRW/eingebettete_systeme_2/UAV_VoiceControl_Benchmark/data/splits/dataset.csv
Gesamt Einträge: 75125


## 10. Verifikation Split

Prüfen ob der Split korrekt ist:
- Verteilung Train / Val / Test insgesamt
- Anzahl pro Klasse und Split
- Kein Datenleck (augmentierte Dateien nur im Training)

In [11]:
from collections import defaultdict

# CSV einlesen
split_counts  = defaultdict(int)           # split → count
class_split   = defaultdict(lambda: defaultdict(int))  # label → split → count
aug_in_val_test = 0

with open(csv_path, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        split_counts[row['split']] += 1
        class_split[row['label']][row['split']] += 1
        if 'augmented' in row['path'] and row['split'] in ('val', 'test'):
            aug_in_val_test += 1

total = sum(split_counts.values())

print('=== Gesamt-Split ===')
for split in ['train', 'val', 'test']:
    n   = split_counts[split]
    pct = n / total * 100
    print(f'  {split:<6}: {n:>6} ({pct:.1f}%)')
print(f'  {"Gesamt":<6}: {total:>6}')

print()
print('=== Klassen-Verteilung ===')
print(f'{"Label":<14} | {"train":>7} | {"val":>7} | {"test":>7} | {"Gesamt":>7}')
print('-' * 52)
all_labels = sorted(class_split.keys())
for label in all_labels:
    tr = class_split[label]['train']
    va = class_split[label]['val']
    te = class_split[label]['test']
    print(f'{label:<14} | {tr:>7} | {va:>7} | {te:>7} | {tr+va+te:>7}')

print()
print(f'Augmentierte Dateien in Val/Test: {aug_in_val_test} {"✓" if aug_in_val_test == 0 else "← PROBLEM!"}')

=== Gesamt-Split ===
  train :  60930 (81.1%)
  val   :   6663 (8.9%)
  test  :   7532 (10.0%)
  Gesamt:  75125

=== Klassen-Verteilung ===
Label          |   train |     val |    test |  Gesamt
----------------------------------------------------
backward       |    3078 |     153 |     165 |    3396
down           |    3134 |     377 |     406 |    3917
forward        |    3095 |     146 |     155 |    3396
go             |    3106 |     372 |     402 |    3880
left           |    3037 |     352 |     412 |    3801
one            |    3140 |     351 |     399 |    3890
right          |    3019 |     363 |     396 |    3778
silence        |      49 |       5 |       5 |      59
stop           |    3111 |     350 |     411 |    3872
two            |    3111 |     345 |     424 |    3880
unknown        |   30102 |    3499 |    3932 |   37533
up             |    2948 |     350 |     425 |    3723

Augmentierte Dateien in Val/Test: 0 ✓
